# Fase 4: nuestra primera arquitectura candidata

Este cuaderno no entrena la red completa. Su objetivo es poder **ver, calcular y defender** la arquitectura que entrenaremos en la GPU. Todo el codigo reutilizable vive en `src/`; el notebook solo lo explica.

## Idea general

```text
[B, 3, 256, 256]
  -> bloque 1 (3 -> 16)    -> [B, 16, 128, 128]
  -> bloque 2 (16 -> 32)   -> [B, 32, 64, 64]
  -> bloque 3 (32 -> 64)   -> [B, 64, 32, 32]
  -> bloque 4 (64 -> 128)  -> [B, 128, 16, 16]
  -> promedio global       -> [B, 128, 1, 1]
  -> flatten + dropout     -> [B, 128]
  -> linear                -> [B]
```

Cada bloque hace dos veces `Conv2d -> BatchNorm -> ReLU` y despues un `MaxPool 2x2`. Las convoluciones conservan alto y ancho; el pooling los divide por dos. A la vez aumentamos canales para permitir mas tipos de patrones aprendidos.

In [1]:
from pathlib import Path
import json
import sys

import pandas as pd

PROJECT = Path.cwd().resolve()
if PROJECT.name == 'notebooks':
    PROJECT = PROJECT.parent
if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))

from src.architectures import BreastPCRNet, parameter_breakdown, trainable_parameter_count
from src.experiment_config import load_experiment_config

config = load_experiment_config(PROJECT / 'configs/experiments/E02_base_normal.json')
model = BreastPCRNet(config.model)
trainable_parameter_count(model)

294129

El resultado anterior debe ser **294.129 parametros entrenables**. Son numeros que la red ajustara mediante backpropagation; no hemos dibujado a mano filtros de lineas o curvas.

In [2]:
pd.DataFrame(
    [{'paso': step.operation, 'forma': str(step.shape)} for step in model.trace_shapes(batch_size=2)]
)

,paso,forma
0,entrada,"(2, 3, 256, 256)"
1,bloque 1,"(2, 16, 128, 128)"
2,bloque 2,"(2, 32, 64, 64)"
3,bloque 3,"(2, 64, 32, 32)"
4,bloque 4,"(2, 128, 16, 16)"
5,global average pooling,"(2, 128, 1, 1)"
6,flatten,"(2, 128)"
7,clasificador,"(2,)"


En estas formas, `B=2` es solo el numero de ejemplos procesados juntos. La red funcionara con otro batch sin cambiar su arquitectura. El orden es siempre `[batch, canales, alto, ancho]`.

In [3]:
details = pd.DataFrame(parameter_breakdown(model))
details

,name,shape,parameters
0,blocks.0.features.0.weight,"[16, 3, 3, 3]",432
1,blocks.0.features.1.weight,[16],16
2,blocks.0.features.1.bias,[16],16
3,blocks.0.features.3.weight,"[16, 16, 3, 3]",2304
4,blocks.0.features.4.weight,[16],16
5,blocks.0.features.4.bias,[16],16
6,blocks.1.features.0.weight,"[32, 16, 3, 3]",4608
7,blocks.1.features.1.weight,[32],32
8,blocks.1.features.1.bias,[32],32
9,blocks.1.features.3.weight,"[32, 32, 3, 3]",9216


## De donde salen los parametros

Una convolucion sin bias contiene `canales_entrada x canales_salida x 3 x 3` pesos. BatchNorm aporta dos parametros aprendibles por canal (escala y desplazamiento). Pooling, ReLU, dropout y promedio global no tienen parametros.

| Parte | Calculo | Parametros |
|---|---:|---:|
| Bloque 1 | `3x16x3x3 + 2x16 + 16x16x3x3 + 2x16` | 2.800 |
| Bloque 2 | `16x32x3x3 + 2x32 + 32x32x3x3 + 2x32` | 13.952 |
| Bloque 3 | `32x64x3x3 + 2x64 + 64x64x3x3 + 2x64` | 55.552 |
| Bloque 4 | `64x128x3x3 + 2x128 + 128x128x3x3 + 2x128` | 221.696 |
| Clasificador | `128 pesos + 1 bias` | 129 |
| **Total** | | **294.129** |

## Campo receptivo

El campo receptivo indica cuantos pixeles originales pueden influir en una activacion. Con convoluciones 3x3 y pooling 2x2 crece asi:

| Punto | Mapa espacial | Campo receptivo | Salto entre posiciones |
|---|---:|---:|---:|
| Entrada | 256 | 1 | 1 |
| Fin bloque 1 | 128 | 6 | 2 |
| Fin bloque 2 | 64 | 16 | 4 |
| Fin bloque 3 | 32 | 36 | 8 |
| Fin bloque 4 | 16 | 76 | 16 |

Antes del promedio global, cada posicion final resume aproximadamente una region de **76x76** pixeles. El promedio global combina las 16x16 posiciones, por lo que la decision final recibe informacion distribuida por toda la imagen.

## Por que promedio global y no un flatten gigante

Aplanar directamente `[128, 16, 16]` produciria 32.768 valores. Conectarlos a una sola salida ya exigiria 32.769 parametros. El promedio global produce solo 128 valores y el clasificador necesita 129 parametros. Esto reduce la posibilidad de memorizar posiciones exactas y obliga a resumir la presencia de patrones a traves de la imagen. Es una hipotesis razonable, no una garantia: la validacion dira si generaliza.

In [4]:
benchmark_path = PROJECT / 'reports/benchmarks/base_local.json'
benchmark = json.loads(benchmark_path.read_text(encoding='utf-8'))
pd.Series({
    'dispositivo': benchmark['device'],
    'batch': benchmark['batch_size'],
    'imagenes_por_segundo': round(benchmark['samples_per_second'], 2),
    'minutos_estimados_por_epoca': round(benchmark['estimated_seconds_per_epoch'] / 60, 2),
    'horas_estimadas_para_50_epocas': round(benchmark['estimated_hours_for_configured_epochs'], 2),
    'recomienda_gpu': benchmark['recommend_gpu'],
})

dispositivo                         cpu
batch                                16
imagenes_por_segundo              25.23
minutos_estimados_por_epoca        5.79
horas_estimadas_para_50_epocas     4.82
recomienda_gpu                     True
dtype: object

## Que vamos a probar primero

1. **E02**: esta arquitectura con BCE normal. Sera nuestra referencia seria.
2. **E03**: exactamente lo mismo, cambiando solo a BCE ponderada para dar mas peso a pCR=1.
3. Compararemos ROC-AUC, PR-AUC, sensibilidad y especificidad **por paciente** en validacion.
4. Solo despues decidiremos que componente arquitectonico merece cambiar.

El benchmark local recomienda la GPU. Por eso este notebook termina aqui y no inicia silenciosamente un entrenamiento de varias horas. El conjunto test permanece sin consultar.